# ISIC experiments on Colab from GitHub
Run cells in order on a Colab GPU. Code is pinned to a tested commit in your fork. MODE selects training or evaluation; MODELS selects which models run. New runs receive unique output directories. For recovery, set RESUME_RUN to a prior run created by the updated code and select models with last.pt. Legacy best.pt files cannot provide an exact resume.

Progress includes phase/batch ETA. Each completed epoch saves last.pt before reporting completion. A disconnect during an epoch loses that unfinished epoch. CPU zero-worker resume is regression-tested for identical weights; persistent worker augmentation states and CUDA numerical determinism are not guaranteed across restarts. Google Drive writes still depend on the remote filesystem; the notebook cannot prevent Colab runtime loss.


In [ ]:
import sys, platform, subprocess, json, hashlib, shutil, time
from pathlib import Path
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
assert sys.platform == "linux" and Path("/content").is_dir(), "Select the remote Colab kernel, not local Python."
assert torch.cuda.is_available(), "Select a Colab GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
subprocess.run(["nvidia-smi"], check=True)


## GitHub source and experiment settings
The original notebook paths are prefilled. `evaluate` uses existing weights; `train` runs the original 10/12-epoch schedules and evaluates the new weights. The installed Colab environment is recorded, not silently replaced with another PyTorch build.


In [ ]:
REPO_URL = "https://github.com/Nandos-tm/PatternAnalysis-2026.git"
COMMIT = "97b148fc714669505711472bd3c5c9be42d9e0c9"
REPO = Path("/content/PatternAnalysis-2026-" + COMMIT[:7])
BASE = Path("/content/drive/MyDrive/COMP3710")
ARCHIVE = BASE / "datasets/ISIC2020/ISIC_2020_Training_JPEG.zip"
IMAGE_DIR = Path("/content/ISIC2020_images")
MODE = "train"  # "evaluate" to check original checkpoints without retraining
CHECKPOINTS = {
    "baseline": BASE / "checkpoints/baseline/best.pt",
    "siamese_aux": BASE / "checkpoints/siamese_aux_head/best.pt",
}
# To resume, set the exact existing run folder containing training/<model>/last.pt.
RESUME_RUN = None  # e.g. BASE / "results/github_verification/<run ID>"
MODELS = ("baseline", "siamese_aux")
LOG_EVERY = 25
WORKERS = 2
BATCH_SIZE = 64  # evaluation only; training retains each script's defaults
RUN_TESTS = True
assert MODE in ("evaluate", "train")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == COMMIT, f"Unexpected checkout: {actual}"
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip(), "Checkout has local changes."
PROJECT = REPO / "recognition/Siamese_Nandos"
SPLITS = PROJECT / "splits"
print("Using:", REPO, actual)


In [ ]:
# Dependency repair used in the prior Colab session.
%pip install jedi==0.20.1


In [ ]:
# Inspect package compatibility before running; preserve the current CUDA build.
from importlib.metadata import version
packages = ["torch", "torchvision", "numpy", "pandas", "Pillow", "scikit-learn", "matplotlib"]
versions = {name: version(name) for name in packages}
print(json.dumps(versions, indent=2))
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)
print(pip_check.stdout or pip_check.stderr or "No dependency conflicts found.")

if pip_check.returncode:
    raise RuntimeError("pip check found dependency conflicts; see output above.")
import pandas as pd
sys.path.insert(0, str(PROJECT))
from split_utils import load_splits
frames = load_splits(SPLITS)
audit = json.loads((SPLITS / "manifest_audit.json").read_text())
for name, details in audit["splits"].items():
    assert hashlib.sha256((SPLITS / f"{name}.csv").read_bytes()).hexdigest() == details["sha256"]
assert hashlib.sha256((PROJECT / "ISIC_2020_Training_Duplicates.csv").read_bytes()).hexdigest() == audit["duplicates"]["sha256"]
print("All manifest hashes and leakage checks passed.")
for name, frame in frames.items():
    print(name, len(frame), "images;", int(frame.target.sum()), "melanomas")
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT), "-v"], cwd=REPO, check=True)


## Connect Google Drive
Complete the Google authorization yourself if prompted. If Drive mounting is unsupported in the VS Code client, open this notebook in the Colab web interface and run it there. Do not paste credentials into notebook code.


In [ ]:
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
assert BASE.is_dir(), f"Project folder not found: {BASE}"
if MODE == "evaluate":
    for name, checkpoint in CHECKPOINTS.items():
        assert checkpoint.is_file(), f"Missing {name} checkpoint: {checkpoint}; update CHECKPOINTS above."
from uuid import uuid4
RUN_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "_" + uuid4().hex[:8]
if RESUME_RUN is not None:
    assert MODE == "train", "Resume applies only to training."
    OUTPUT = Path(RESUME_RUN)
    assert OUTPUT.is_dir(), f"Resume directory missing: {OUTPUT}"
    for model_name in MODELS:
        assert (OUTPUT / "training" / model_name / "last.pt").is_file(), (
            f"No resumable checkpoint for {model_name}. Select only models with last.pt in MODELS; "
            "best.pt from the old code cannot resume exactly.")
else:
    OUTPUT = BASE / "results/github_verification" / RUN_ID
    OUTPUT.mkdir(parents=True, exist_ok=False)
metadata = {"commit": COMMIT, "mode": MODE, "packages": versions,
            "python": sys.version, "gpu": torch.cuda.get_device_name(0),
            "cuda_build": torch.version.cuda, "manifest_audit": audit,
            "inference_precision": "FP16 CUDA autocast (predict.py)",
            "workers": WORKERS, "evaluation_batch_size": BATCH_SIZE,
            "models": list(MODELS), "resume": RESUME_RUN is not None}
(OUTPUT / f"environment_{RUN_ID}.json").write_text(json.dumps(metadata, indent=2))
print("Results will be saved to:", OUTPUT)


## Extract required JPEGs to local Colab storage
Evaluation needs validation/test images. Training also needs training images. Extraction writes only normalized JPEG filenames and checks archive completeness, duplicate IDs and available space.


In [ ]:
from zipfile import ZipFile
from dataset import index_images
parts = ("train", "val", "test") if MODE == "train" else ("val", "test")
needed = {name for part in parts for name in frames[part].image_name}
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
paths = index_images(IMAGE_DIR)
missing = needed - paths.keys()
if missing:
    assert ARCHIVE.is_file(), f"Missing archive: {ARCHIVE}"
    with ZipFile(ARCHIVE) as archive:
        members = {}
        for member in archive.infolist():
            key = Path(member.filename).stem
            if Path(member.filename).suffix.lower() in (".jpg", ".jpeg") and key in missing:
                if key in members:
                    raise ValueError(f"Duplicate archive image ID: {key}")
                members[key] = member
        assert missing <= members.keys(), f"Archive lacks {len(missing - members.keys())} required images"
        required_bytes = sum(m.file_size for m in members.values())
        assert shutil.disk_usage(IMAGE_DIR).free > required_bytes + 2 * 1024**3, "Insufficient local disk space"
        for i, (key, member) in enumerate(members.items(), 1):
            target = IMAGE_DIR / (key + Path(member.filename).suffix.lower())
            temporary = target.with_suffix(".partial")
            with archive.open(member) as source, temporary.open("wb") as destination:
                shutil.copyfileobj(source, destination)
            assert temporary.stat().st_size == member.file_size
            temporary.replace(target)
            if i % 1000 == 0:
                print(f"Extracted {i}/{len(members)}", flush=True)
paths = index_images(IMAGE_DIR)
assert needed <= paths.keys()
print("Required JPEGs available:", len(needed))


## Launch and log the experiment
Logs stream into the notebook and are saved to Drive. With `MODE='train'`, both models are trained from ImageNet initialization in new folders before held-out evaluation. With `MODE='evaluate'`, the original checkpoints are used. The validation threshold policy is fixed at 80% specificity.


In [ ]:
import os, signal

# Repeat checks immediately before launching, even if setup cells ran out of order.
def preflight(parts):
    checked = load_splits(SPLITS)
    indexed = index_images(IMAGE_DIR)
    required = {image_id for part in parts for image_id in checked[part].image_name}
    missing = required - indexed.keys()
    if missing:
        raise FileNotFoundError(f"{len(missing)} required JPEGs are missing. Finish the extraction cell first. Examples: {sorted(missing)[:5]}")
    assert torch.cuda.is_available(), "CUDA is unavailable."
    print(f"Launch preflight passed: {len(required)} images and leakage audit.", flush=True)


def run_script(script, arguments, log_name):
    preflight(("train", "val") if script == "train.py" else ("val", "test"))
    command = [sys.executable, "-u", str(PROJECT / script), *map(str, arguments)]
    log_path = OUTPUT / f"{Path(log_name).stem}_{time.strftime('%H%M%S', time.gmtime())}_{uuid4().hex[:8]}.log"
    print("Running:", command, "\nLog:", log_path, flush=True)
    # Append-only attempt history; each attempt gets its own log.
    with (OUTPUT / "commands.jsonl").open("a") as record:
        record.write(json.dumps({"command": command, "log": str(log_path)}) + "\n")
    with log_path.open("x") as log:
        log.write(json.dumps(command) + "\n")
        log.flush()
        process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1,
                                   env={**os.environ, "PYTHONUNBUFFERED": "1"},
                                   start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            # Colab is Linux: terminate the entire group, including DataLoader workers.
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                pass
            # Kill any descendants that survived after the parent exited.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
            raise
        finally:
            process.stdout.close()
        if code:
            raise subprocess.CalledProcessError(code, command)

if MODE == "train":
    for model_name in MODELS:
        train_output = OUTPUT / "training" / model_name
        run_script("train.py", ["--model", model_name, "--splits", SPLITS,
                   "--images", IMAGE_DIR, "--output", train_output,
                   "--workers", WORKERS, "--device", "cuda", "--log-every", LOG_EVERY,
                   *(["--resume"] if RESUME_RUN is not None else [])], f"{model_name}_train.log")
        CHECKPOINTS[model_name] = train_output / "best.pt"
else:
    print("Using original checkpoints; no retraining.")


In [ ]:
checkpoint_hashes = {}
for model_name in MODELS:
    checkpoint = CHECKPOINTS[model_name]
    checkpoint_hashes[model_name] = {"path": str(checkpoint), "sha256": hashlib.sha256(checkpoint.read_bytes()).hexdigest()}
    run_script("predict.py", ["--model", model_name, "--checkpoint", checkpoint,
               "--splits", SPLITS, "--images", IMAGE_DIR, "--output", OUTPUT / f"{model_name}_{RUN_ID}",
               "--batch-size", BATCH_SIZE, "--workers", WORKERS, "--device", "cuda",
               "--threshold-policy", "val_specificity80_max_sensitivity"], f"{model_name}_evaluation.log")
(OUTPUT / f"checkpoints_{RUN_ID}.json").write_text(json.dumps(checkpoint_hashes, indent=2))
summary = pd.concat([pd.read_csv(OUTPUT / f"{name}_{RUN_ID}" / "test_metrics.csv") for name in MODELS], ignore_index=True)
summary.to_csv(OUTPUT / f"comparison_{RUN_ID}.csv", index=False)
display(summary)
print("Experiment completed. Results:", OUTPUT)
